# Laser Microphone - Data Exploration

This notebook is for **understanding what the data looks like** before/while we build the model. It loads one spoken-digit sample and visualizes it three ways:

1. **Waveform** - the raw signal (amplitude over time)
2. **MFCC** - the MAIN feature we feed to the LSTM
3. **Mel spectrogram** - the BACKUP/comparison feature (and good for visualization)

> Run `python scripts/download_fsdd.py` first so there are WAV files in `data/raw/`.

**Laser note:** later, swap the loaded file for a laser capture (WAV or CSV via `preprocess.load_waveform_from_array`). The plots below will then show what the laser signal looks like.

In [ ]:
# Make the src/ modules importable from the notebook.
import sys, os
sys.path.append(os.path.abspath('../src'))

import torch
import matplotlib.pyplot as plt

from config import RAW_DATA_DIR, SAMPLE_RATE, N_MFCC
from preprocess import load_audio, preprocess_waveform
from features import extract_mfcc, extract_mel_spectrogram, extract_spectrogram
print('Imports OK. Sample rate =', SAMPLE_RATE, 'Hz')

In [ ]:
# Pick the first WAV file in data/raw/.
from pathlib import Path
wavs = sorted(Path(RAW_DATA_DIR).rglob('*.wav'))
assert wavs, 'No WAV files found. Run: python scripts/download_fsdd.py'
sample_path = wavs[0]
print('Using:', sample_path.name)

raw = load_audio(sample_path)            # mono, resampled (raw, not trimmed)
clean = preprocess_waveform(raw)         # normalized, silence-trimmed, fixed length
print('raw length   :', raw.shape)
print('clean length :', clean.shape)

In [ ]:
# 1) Waveform (before vs after preprocessing)
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=False)
axes[0].plot(raw.numpy()); axes[0].set_title('Raw waveform')
axes[1].plot(clean.numpy()); axes[1].set_title('Preprocessed (trimmed + normalized + padded)')
for ax in axes: ax.set_xlabel('sample'); ax.set_ylabel('amplitude')
plt.tight_layout(); plt.show()

In [ ]:
# 2) MFCC - the MAIN feature for the LSTM. Shape is (time_steps, n_mfcc).
mfcc = extract_mfcc(clean)
print('MFCC shape:', tuple(mfcc.shape), '-> (time_steps, n_mfcc); LSTM input_size =', N_MFCC)
plt.figure(figsize=(10, 4))
plt.imshow(mfcc.T.numpy(), origin='lower', aspect='auto')
plt.title('MFCC'); plt.xlabel('time frame'); plt.ylabel('MFCC coefficient'); plt.colorbar()
plt.tight_layout(); plt.show()

In [ ]:
# 3) Mel spectrogram (backup feature) and plain spectrogram (visualization).
mel = extract_mel_spectrogram(clean)
spec = extract_spectrogram(clean)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(mel.numpy(), origin='lower', aspect='auto'); axes[0].set_title('Mel spectrogram (dB)')
axes[1].imshow(spec.numpy(), origin='lower', aspect='auto'); axes[1].set_title('Spectrogram (dB)')
for ax in axes: ax.set_xlabel('time frame'); ax.set_ylabel('frequency bin')
plt.tight_layout(); plt.show()

## Takeaways
- The **MFCC** is a compact `(time_steps, n_mfcc)` matrix - a short sequence of feature vectors. That is exactly what the LSTM reads, one time step at a time.
- Preprocessing removes silence and normalizes loudness so the model focuses on the digit's sound shape.
- To explore laser data later, replace the loaded file and re-run; everything downstream is identical.